<div align="center" style="font-size: 2.5em; font-weight: bold; margin-bottom: 10px;">Diplomski rad — GraphRAG pristup</div>

<div align="right" style="font-style: italic; color: #666;">by Zlatko Pračić</div>

## 1. Postavljanje okruženja

za sljedećih 5 ćelija vidi detaljno objašnjenje u Vanilla notebooku, ovdje se nalazi identičan kod (dodane su stavke za potrebe ove vrste chatbota)

In [ ]:
!pip install -q -U bitsandbytes transformers accelerate sentencepiece
!pip install -q -U sentence-transformers
!pip install -q python-docx pdfplumber==0.11.0
!pip install -q neo4j tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 162.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 121.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 10.2 MB/s eta 0:00:00


In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig, GenerationConfig
from google.colab import userdata
from huggingface_hub import login
from sentence_transformers import SentenceTransformer
from neo4j import GraphDatabase
from tqdm import tqdm
from docx import Document as DocxDocument
import pdfplumber
from pathlib import Path
from copy import deepcopy
import torch
import time
import json
import os
import re
import logging
import pandas as pd

In [ ]:
# model_name = "Qwen/Qwen2.5-14B-Instruct"
model_name = "utter-project/EuroLLM-22B-Instruct-2512"

from google.colab import drive
drive.mount('/content/drive')

os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

print(f"Model: {model_name}")
print(f"Čitanje s Drivea: {os.environ['HF_HOME']}")

Mounted at /content/drive
Model: utter-project/EuroLLM-22B-Instruct-2512
Čitanje s Drivea: /content/drive/MyDrive/hf_cache


In [ ]:
token = userdata.get('HF_TOKEN')
login(token)

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

pipe = pipeline(
    "text-generation",
    model=model_name,
    dtype=torch.float16,
    device_map="auto",
    model_kwargs={"quantization_config": quantization_config}
)

pipe.model.generation_config.max_length = 8192

config.json:   0%|          | 0.00/680 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/40.3k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

...

tokenizer.model: reconstructing file:   0%|          |  0.00B / 2.41MB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 15.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

## 2. Konfiguracija Neo4j baze podataka

Definiramo parametre veze s Neo4j bazom i konfiguriramo retrieval.

In [ ]:
NEO4J_URI      = 'neo4j+s://ad1afaa8.databases.neo4j.io'
NEO4J_USER     = 'ad1afaa8'
NEO4J_PASSWORD = userdata.get('Neo4j')

GRAPH_TOP_K = {
    'jednostavno':                5,
    'kompleksno_jedan_dokument':  8,
    'kompleksno_vise_dokumenata': 12,
}
GRAPH_TOP_K_DEFAULT = 5

GRAPH_VECTOR_POOL = 30

GRAPH_PER_DOC_CAP = 3

GRAPH_MAX_CHARS_PO_CLANKU = 4000

GRAPH_REL_WEIGHTS = {
    'RAZRADJUJE':  1.00,
    'ISTA_TEMA':   0.95,
    'UPUCUJE_NA':  0.85,
    'SPOMINJE':    0.70,
}

print('\u2713 Neo4j i retrieval konfiguracija ucitana')


✓ Neo4j i retrieval konfiguracija ucitana


### Konekcija s Neo4j i kreiranje sheme grafa

Uspostavljamo vezu s Neo4j bazom i kreiramo:
- **Unique constraints** za čvorove Zakon i Clanak
- **Vektorski indeks** za semantičko pretraživanje embeddings

[Neo4j Python Driver](https://neo4j.com/docs/python-manual/current/)

In [ ]:
neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def neo4j_run(query, params=None):
    with neo4j_driver.session() as session:
        return list(session.run(query, params or {}))

_transformers_logger = logging.getLogger("transformers.modeling_utils")
_prev_level = _transformers_logger.level
_transformers_logger.setLevel(logging.ERROR)

embed_model = SentenceTransformer("intfloat/multilingual-e5-large")

_transformers_logger.setLevel(_prev_level)

EMBEDDING_DIM = embed_model.get_embedding_dimension()  # 1024 za e5-large

SCHEMA_QUERIES = [
    'CREATE CONSTRAINT IF NOT EXISTS FOR (z:Zakon) REQUIRE z.id IS UNIQUE',
    'CREATE CONSTRAINT IF NOT EXISTS FOR (c:Clanak) REQUIRE c.id IS UNIQUE',
    'CREATE CONSTRAINT IF NOT EXISTS FOR (e:Entitet) REQUIRE e.naziv IS UNIQUE',
    f'''CREATE VECTOR INDEX clanak_embedding IF NOT EXISTS
    FOR (c:Clanak) ON (c.embedding)
    OPTIONS {{
        indexConfig: {{
            `vector.dimensions`: {EMBEDDING_DIM},
            `vector.similarity_function`: 'cosine'
        }}
    }}''',
]

for q in SCHEMA_QUERIES:
    try:
        neo4j_run(q)
    except Exception as e:
        print(f'  (info) {e}')

neo4j_driver.verify_connectivity()
print(f'✓ Neo4j konekcija OK | Embedding dimenzija: {EMBEDDING_DIM}')

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

✓ Neo4j konekcija OK | Embedding dimenzija: 1024


## 3. Sistemska poruka i funkcija generacije

Definiramo sistemsku poruku i funkciju za generaciju odgovora LLM modelom.

In [ ]:
SYSTEM_PROMPT = (
"Ti si pravni stručnjak specijaliziran za zakonodavstvo Republike Hrvatske, posebno za Zakon o obrani, Zakon o službi u Oružanim snagama RH i pripadajuće pravilnike."
"\n\nPravila odgovaranja:"
"\n1. Jezik odgovora: hrvatski, latinica."
"\n2. Navedi konkretne članke zakona, brojčane vrijednosti i nadležna tijela."
"\n3. Ako je dostupan kontekst, koristi SAMO informacije iz njega."
"\n4. Ako kontekst ne sadržava odgovor, odgovori na temelju općeg znanja i dodaj: 'Napomena: ova informacija nije pronađena u dostupnim dokumentima."
"\n5. Odgovor neka bude koncizan i strukturiran — bez nepotrebnih uvoda."
)

In [ ]:
csv_path = '/content/drive/My Drive/diplomskiRad/rezultati.csv'

In [ ]:
def generate_response(messages, max_new_tokens=512, temperature=0.5, do_sample=True):
    final_messages = list(messages)

    model_max_length = getattr(pipe.tokenizer, 'model_max_length', 8192)
    if model_max_length > 100000:
        model_max_length = 8192

    input_text = pipe.tokenizer.apply_chat_template(final_messages, tokenize=False)
    input_token_count = len(pipe.tokenizer.encode(input_text))
    max_allowed_input = model_max_length - max_new_tokens

    if input_token_count > max_allowed_input:
        print(f"Upozorenje: input ({input_token_count} tokena) prelazi limit ({max_allowed_input}). Skraćujem kontekst.")
        if final_messages and final_messages[0]["role"] == "system":
            system_content = final_messages[0]["content"]
            while input_token_count > max_allowed_input and "\n\n---\n\n" in system_content:
                last_chunk_start = system_content.rfind("\n\n---\n\n")
                system_content = system_content[:last_chunk_start]
                final_messages[0]["content"] = system_content
                input_text = pipe.tokenizer.apply_chat_template(final_messages, tokenize=False)
                input_token_count = len(pipe.tokenizer.encode(input_text))
            print(f"  Skraćeno na {input_token_count} tokena.")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    gen_config = deepcopy(pipe.model.generation_config)
    gen_config.max_new_tokens = max_new_tokens
    gen_config.temperature = temperature
    gen_config.do_sample = do_sample
    gen_config.repetition_penalty = 1.1

    vrijeme_upita = time.time()

    response = pipe(final_messages, generation_config=gen_config)

    answer = ""
    for message in response[0]['generated_text']:
        if message['role'] == 'assistant':
            answer = message['content']
            break

    stripped_answer = answer.strip()
    if not re.search(r'[.!?]$', stripped_answer) and len(stripped_answer.split()) > 5:
        last_punctuation_index = -1
        for i in range(len(stripped_answer) - 1, -1, -1):
            if stripped_answer[i] in ['.', '?', '!']:
                last_punctuation_index = i
                break
        if last_punctuation_index != -1:
            answer = stripped_answer[:last_punctuation_index + 1]
        else:
            words = stripped_answer.split()
            answer = ' '.join(words[:-1]) + '...' if len(words) > 1 else (words[0] if words else '')
    else:
        answer = stripped_answer

    vrijeme_odgovora = time.time()
    ukupno_vrijeme = vrijeme_odgovora - vrijeme_upita

    print("Generirani odgovor:")
    print(answer)
    print(f"Ukupno vrijeme za odgovor: {ukupno_vrijeme:.2f} sekundi")

    return answer, ukupno_vrijeme

## 4. NER model za ekstrakciju entiteta

Za prepoznavanje imenskih entiteta (osobe, organizacije, mjesta) iz pravnih tekstova koristimo višejezični BERTić model.

[BERTić NER model](https://huggingface.co/classla/bcms-bertic-ner)

In [ ]:
graph_ner_pipe = pipeline(
    'ner',
    model='classla/bcms-bertic-ner',
    aggregation_strategy='simple',
    device=0,
)

def graph_extract_entities(text: str) -> list:
    if not text or len(text.strip()) < 10:
        return []
    try:
        entities, seen = [], set()
        for start in range(0, len(text), 384):
            prozor = text[start:start + 512]
            if len(prozor.strip()) < 10:
                continue
            for r in graph_ner_pipe(prozor):
              naziv = r['word'].strip()
              if '##' in naziv:                       # subword token iz BERTica
                    continue
              if naziv and naziv not in seen and len(naziv) > 2:
                    entities.append({'naziv': naziv, 'tip': r['entity_group']})
                    seen.add(naziv)
        return entities
    except Exception:
        return []

print('✓ NER model ucitan')

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/352 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/231k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

✓ NER model ucitan


## 5. Parser pravnih dokumenata

Parser čita .docx datoteke i izvlači strukturu zakona po člancima, stavcima, upućivanjima i obvezama.

In [ ]:
CLANAK_RE      = re.compile(r'^[\u010c\u010dCc]lanak\s+(\d+[a-z]?)\.?(?:\s|$)', re.IGNORECASE)
STAVAK_PATTERN = re.compile(r'^\((\d+)\)\s+')

UPUTA_PATTERN  = re.compile(
    r'[\u010c\u010dCc]lank[ua]?\s+(\d+[a-z]?)\.?'
    r'(?:\s+(ovog[a]?\s+(?:Pravilnika|Zakona)'
    r'|Ustav[a]?(?:\s+Republike\s+Hrvatske)?'
    r'|Zakona\s+o\s+obrani'
    r'|Zakona\s+o\s+slu\u017ebi[a-z\u0107\u010d\u0111\u0161\u017e\s]*?snagama'
    r'|Pravilnika\s+o\s+[a-z\u0107\u010d\u0111\u0161\u017e\s]+?))?',
    re.IGNORECASE
)

def _ciljni_akt(fraza):
    if not fraza:                       return None
    f = fraza.lower()
    if 'ovog' in f:                     return None
    if 'ustav' in f:                    return 'ustav'
    if 'slu\u017ebi' in f or 'sluzbi' in f: return 'sluzba_osrh'
    if 'obrani' in f:                   return 'obrana'
    if 'pravilnik' in f:                return 'pravilnik'
    return None

FALLBACK_CHUNK_LEN = 1200

def _klasificiraj_obvezu(tekst: str):
    t = tekst.lower()
    if any(k in t for k in ['du\u017ean', 'duzan', 'obvezan', 'obvezna', 'mora', 'obvezuje se']):
        return 'OBVEZA'
    if any(k in t for k in ['zabranjeno', 'nije dopu\u0161teno', 'nije dopusteno', 'ne smije']):
        return 'ZABRANA'
    if any(k in t for k in ['ima pravo', 'ovla\u0161ten', 'ovlasten', 'mo\u017ee', 'moze']):
        return 'PRAVO'
    return None

def _fallback_pseudo_clanci(tekst: str, prefix: str):
    """Tekst izvan obrasca clanka rezemo u pseudo-clanke (fallback,
    identican duh RAG chunkera) da nista ne ispadne iz grafa."""
    tekst = ' '.join(tekst.split())
    if len(tekst) < 200:
        return []
    out, i, n = [], 0, 1
    while i < len(tekst):
        chunk = tekst[i:i + FALLBACK_CHUNK_LEN]
        out.append({'broj': f'{prefix}{n}', 'tekst': chunk,
                    'stavci': [], 'upucivanja': [], 'obveze': []})
        i += FALLBACK_CHUNK_LEN
        n += 1
    return out

def _parse_paragraphs(paragraphs, naziv):
    """Zajednicka logika za docx i pdf: paragrafe pretvara u clanke,
    uz fallback za tekst prije prvog clanka i dokumente bez clanaka."""
    clanaci, current = [], None
    pre_text = []

    for tekst in paragraphs:
        tekst = tekst.strip()
        if not tekst:
            continue

        m = CLANAK_RE.match(tekst)
        if m:
            if current:
                clanaci.append(current)
            current = {'broj': m.group(1), 'tekst': '', 'stavci': [],
                       'upucivanja': [], 'obveze': []}
            continue

        if current is None:
            pre_text.append(tekst)
            continue

        if STAVAK_PATTERN.match(tekst):
            current['stavci'].append(tekst)
        else:
            current['tekst'] += ' ' + tekst

        for um in UPUTA_PATTERN.finditer(tekst):
            br  = um.group(1)
            akt = _ciljni_akt(um.group(2))
            ref = (br, akt)
            if br and ref not in current['upucivanja']:
                current['upucivanja'].append(ref)

        tip = _klasificiraj_obvezu(tekst)
        if tip:
            current['obveze'].append({'tip': tip, 'tekst': tekst[:300]})

    if current:
        clanaci.append(current)

    pseudo = _fallback_pseudo_clanci(' '.join(pre_text), 'pre')

    if not clanaci and not pseudo:
        cijeli = ' '.join(paragraphs)
        pseudo = _fallback_pseudo_clanci(cijeli, 'chunk')

    clanaci = pseudo + clanaci
    print(f'  Parsiran: {naziv} \u2014 {len(clanaci)} clanaka'
          f' (od toga {len(pseudo)} fallback)')
    return {'naziv': naziv, 'clanaci': clanaci}

def graph_parse_docx(filepath: str) -> dict:
    doc = DocxDocument(filepath)
    naziv = Path(filepath).stem.replace('_', ' ')
    for para in doc.paragraphs[:15]:
        sn = para.style.name.lower()
        if ('heading' in sn or 'naslov' in sn or 'title' in sn) and len(para.text.strip()) > 10:
            naziv = para.text.strip()
            break
    return _parse_paragraphs([p.text for p in doc.paragraphs], naziv)

def graph_parse_pdf(filepath: str) -> dict:
    import pdfplumber
    naziv = Path(filepath).stem.replace('_', ' ')
    lines = []
    try:
        with pdfplumber.open(filepath) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    lines.extend(page_text.split("\n"))
    except Exception as e:
        print(f'  Greska pri citanju PDF-a {filepath}: {e}')
        return {'naziv': naziv, 'clanaci': []}

    paragraphs, current_para = [], []
    for line in lines:
        line = line.strip()
        if not line:
            if current_para:
                paragraphs.append(" ".join(current_para))
                current_para = []
            continue
        if CLANAK_RE.match(line):
            if current_para:
                paragraphs.append(" ".join(current_para))
                current_para = []
            paragraphs.append(line)
        elif STAVAK_PATTERN.match(line):
            if current_para:
                paragraphs.append(" ".join(current_para))
            current_para = [line]
        else:
            current_para.append(line) if current_para else current_para.append(line)
    if current_para:
        paragraphs.append(" ".join(current_para))

    return _parse_paragraphs(paragraphs, naziv)

print('\u2713 GraphRAG docx/pdf parser ucitan (s fallbackom)')


✓ GraphRAG docx/pdf parser ucitan (s fallbackom)


## 6. Ingestija zakona u Neo4j

Za svaki članak zakona:
1. Kreiramo čvor `(:Clanak)` s tekstom, brojem i vektorskim embeddingom
2. Kreiramo čvorove `(:Stavak)` s relacijom `[:IMA_STAVAK]`
3. Dodajemo relacije `[:UPUCUJE_NA]` između članaka koji se međusobno referiraju
4. Ekstrahiramo entitete (NER) i dodajemo `(:Entitet)` čvorove
5. Kreiramo `(:Obveza)` čvorove za klasificirane obveze

In [ ]:
_AKT_ID_CACHE = {}

def _rijesi_zakon_id(akt, vlastiti_zakon_id):
    if akt is None:
        return vlastiti_zakon_id

    if not _AKT_ID_CACHE:
        try:
            for row in neo4j_run('MATCH (z:Zakon) RETURN z.id AS id, z.naziv AS naziv'):
                naziv = (row['naziv'] or '').lower()
                zid   = row['id']
                if 'ustav' in naziv:
                    _AKT_ID_CACHE['ustav'] = zid
                elif 'slu\u017ebi' in naziv or 'sluzbi' in naziv:
                    _AKT_ID_CACHE['sluzba_osrh'] = zid
                elif 'obrani' in naziv:
                    _AKT_ID_CACHE['obrana'] = zid
        except Exception as e:
            print(f'  (info) _rijesi_zakon_id kes: {e}')

    return _AKT_ID_CACHE.get(akt, vlastiti_zakon_id)

def graph_ensure_schema():
    """Idempotentno osigurava constraintove i vektorske indekse."""
    dim = embed_model.get_embedding_dimension()
    queries = [
        'CREATE CONSTRAINT IF NOT EXISTS FOR (z:Zakon) REQUIRE z.id IS UNIQUE',
        'CREATE CONSTRAINT IF NOT EXISTS FOR (c:Clanak) REQUIRE c.id IS UNIQUE',
        'CREATE CONSTRAINT IF NOT EXISTS FOR (s:Stavak) REQUIRE s.id IS UNIQUE',
        'CREATE CONSTRAINT IF NOT EXISTS FOR (e:Entitet) REQUIRE e.naziv IS UNIQUE',
        f'''CREATE VECTOR INDEX clanak_embedding IF NOT EXISTS
        FOR (c:Clanak) ON (c.embedding)
        OPTIONS {{
            indexConfig: {{
                `vector.dimensions`: {dim},
                `vector.similarity_function`: 'cosine'
            }}
        }}''',
        f'''CREATE VECTOR INDEX stavak_embedding IF NOT EXISTS
        FOR (s:Stavak) ON (s.embedding)
        OPTIONS {{
            indexConfig: {{
                `vector.dimensions`: {dim},
                `vector.similarity_function`: 'cosine'
            }}
        }}''',
    ]
    for q in queries:
        try:
            neo4j_run(q)
        except Exception as e:
            print(f'  (info) {e}')
    print(f'\u2713 Shema osigurana (indeksi clanak_embedding i stavak_embedding, dim={dim})')

def graph_embed(text: str) -> list:
    # Pun tekst — e5-large sam reze na 512 tokena pri enkodiranju
    vec = embed_model.encode(['passage: ' + text], normalize_embeddings=True)
    return vec[0].tolist()

import unicodedata as _ud

_PADEZNI_NASTAVCI = ('oga', 'ome', 'ima', 'ama', 'ega', 'emu',
                     'og', 'om', 'im', 'em', 'a', 'e', 'i', 'u', 'o')

def _kanon_rijec(w: str) -> str:
    for suf in _PADEZNI_NASTAVCI:
        if len(w) > len(suf) + 3 and w.endswith(suf):
            return w[:-len(suf)]
    return w

def normaliziraj_entitet(naziv: str) -> str:
    naziv = (naziv or '').strip().lower()
    naziv = ''.join(c for c in naziv if c.isalnum() or c.isspace())
    naziv = ' '.join(naziv.split())
    if not naziv:
        return ''
    return ' '.join(_kanon_rijec(w) for w in naziv.split())

# Genericki pojmovi koji se pojavljuju u svakom clanku svakog zakona.
# Bez filtriranja svaki par clanaka dijeli 2+ takva entiteta -> ISTA_TEMA
# veza bez ikakve tematske vrijednosti (2579 veza u prvoj verziji).
STOP_ENTITETI = {
    'republik hrvatsk', 'republic hrvatskoj', 'hrvatsk', 'republik',
    'oružanih snag', 'oružan snag', 'oruž', 'snag',
    'ministarstv obran', 'hrvatsk sabor', 'narodn novin',
    'vlad republik hrvatsk', 'vlad', 'europsk unij',
    'nim snag', 'ne snag',
}

def graph_ingest_zakon(zakon_data: dict, zakon_id: str):
    graph_ensure_schema()
    naziv = zakon_data['naziv']
    neo4j_run('MERGE (z:Zakon {id: $id}) SET z.naziv = $naziv',
              {'id': zakon_id, 'naziv': naziv})

    for cl in tqdm(zakon_data['clanaci'], desc=f'  Ingestija: {naziv[:30]}'):
        clanak_id = f'{zakon_id}_cl_{cl["broj"]}'
        full_text = (cl['tekst'].strip() + ' ' + ' '.join(cl['stavci'])).strip()
        if not full_text:
            continue

        embedding = graph_embed(full_text)

        neo4j_run('''
            MERGE (c:Clanak {id: $id})
            SET c.broj = $broj, c.tekst = $tekst,
                c.zakon_id = $zakon_id, c.embedding = $emb
            WITH c
            MATCH (z:Zakon {id: $zakon_id})
            MERGE (z)-[:SADRZI]->(c)
        ''', {'id': clanak_id, 'broj': cl['broj'], 'tekst': full_text,
              'zakon_id': zakon_id, 'emb': embedding})

        for i, stavak in enumerate(cl['stavci']):
            neo4j_run('''
                MERGE (s:Stavak {id: $id})
                SET s.tekst = $tekst, s.redoslijed = $r, s.embedding = $emb
                WITH s MATCH (c:Clanak {id: $cid})
                MERGE (c)-[:IMA_STAVAK]->(s)
            ''', {'id': f'{clanak_id}_s{i}', 'tekst': stavak, 'r': i,
                  'cid': clanak_id, 'emb': graph_embed(stavak)})

        for target_br, akt in cl['upucivanja']:
            ciljni_zakon = _rijesi_zakon_id(akt, zakon_id)
            neo4j_run('''
                MERGE (t:Clanak {id: $tid})
                ON CREATE SET t.broj = $tbr, t.zakon_id = $zid
                WITH t MATCH (src:Clanak {id: $sid})
                MERGE (src)-[:UPUCUJE_NA]->(t)
            ''', {'tid': f'{ciljni_zakon}_cl_{target_br}', 'tbr': target_br,
                  'zid': ciljni_zakon, 'sid': clanak_id})

        for ent in graph_extract_entities(full_text):
            kanon = normaliziraj_entitet(ent['naziv'])
            if not kanon or kanon in STOP_ENTITETI or len(kanon) < 6:
                continue
            neo4j_run('''
                MERGE (e:Entitet {naziv: $kanon})
                SET e.tip = $tip, e.povrsinski = $povrsinski
                WITH e MATCH (c:Clanak {id: $cid})
                MERGE (c)-[:SPOMINJE]->(e)
            ''', {'kanon': kanon, 'tip': ent['tip'],
                  'povrsinski': ent['naziv'], 'cid': clanak_id})

        for ob in cl['obveze']:
            neo4j_run('''
                CREATE (o:Obveza {tekst: $tekst, tip: $tip})
                WITH o MATCH (c:Clanak {id: $cid})
                MERGE (c)-[:SADRZI_OBVEZU]->(o)
            ''', {'tekst': ob['tekst'], 'tip': ob['tip'], 'cid': clanak_id})

    print(f'\u2713 Zakon "{naziv}" upisan u Neo4j')

print('\u2713 Ingestija funkcije ucitane')


✓ Ingestija funkcije ucitane


### Učitavanje zakona iz Google Drivea u Neo4j

Koristimo .docx dokumente iz Google Drivea, parsiramo ih i upisujemo u Neo4j.

In [ ]:
neo4j_run('MATCH (n) DETACH DELETE n')
print('✓ Baza očišćena — svi čvorovi i veze obrisani.')

✓ Baza očišćena — svi čvorovi i veze obrisani.


In [ ]:
graph_docs_dir = '/content/drive/My Drive/diplomskiRad/Dokumenti'

if os.path.exists(graph_docs_dir):
    print(f'Ingestiram dokumente iz: {graph_docs_dir}\n')
    for file in os.listdir(graph_docs_dir):
        if file.endswith('.docx'):
            file_path = os.path.join(graph_docs_dir, file)
            zakon_id = Path(file).stem.lower().replace(' ', '_')[:50]
            zakon_data = graph_parse_docx(file_path)
            graph_ingest_zakon(zakon_data, zakon_id)
        elif file.endswith('.pdf'):
            file_path = os.path.join(graph_docs_dir, file)
            zakon_id = Path(file).stem.lower().replace(' ', '_')[:50]
            zakon_data = graph_parse_pdf(file_path)
            graph_ingest_zakon(zakon_data, zakon_id)
    print('\n✓ Svi zakoni upisani u Neo4j')
else:
    print(f'Mapa nije pronađena: {graph_docs_dir}')

Ingestiram dokumente iz: /content/drive/My Drive/diplomskiRad/Dokumenti

  Parsiran: Pravilnik o temeljnom vojnom osposobljavanju — 23 clanaka (od toga 1 fallback)
✓ Shema osigurana (indeksi clanak_embedding i stavak_embedding, dim=1024)


  Ingestija: Pravilnik o temeljnom vojnom o: 100%|██████████| 23/23 [00:17<00:00,  1.35it/s]


✓ Zakon "Pravilnik o temeljnom vojnom osposobljavanju" upisan u Neo4j
  Parsiran: Ustav Republike Hrvatske 2018 — 154 clanaka (od toga 5 fallback)
✓ Shema osigurana (indeksi clanak_embedding i stavak_embedding, dim=1024)


  Ingestija: Ustav Republike Hrvatske 2018: 100%|██████████| 154/154 [01:18<00:00,  1.96it/s]


✓ Zakon "Ustav Republike Hrvatske 2018" upisan u Neo4j
  Parsiran: Zakon o obrani 2025 — 143 clanaka (od toga 2 fallback)
✓ Shema osigurana (indeksi clanak_embedding i stavak_embedding, dim=1024)


  Ingestija: Zakon o obrani 2025: 100%|██████████| 143/143 [03:16<00:00,  1.38s/it]


✓ Zakon "Zakon o obrani 2025" upisan u Neo4j
  Parsiran: Zakon o sluzbi u Oruzanim snagama Republike Hrvatske 2025 — 280 clanaka (od toga 2 fallback)
✓ Shema osigurana (indeksi clanak_embedding i stavak_embedding, dim=1024)


  Ingestija: Zakon o sluzbi u Oruzanim snag: 100%|██████████| 280/280 [04:55<00:00,  1.05s/it]

✓ Zakon "Zakon o sluzbi u Oruzanim snagama Republike Hrvatske 2025" upisan u Neo4j

✓ Svi zakoni upisani u Neo4j


## 7. Derivacija ISTA_TEMA veza i dijagnostika grafa

Automatski gradimo međudokumentne `ISTA_TEMA` veze iz zajedničkih (normaliziranih) entiteta i ispisujemo koliko je `UPUCUJE_NA` zaista međudokumentno. Pokrenuti nakon ingestije zakona (odjeljak 6. — potrebno je da graf već sadrži čvorove `Clanak` i `Entitet`).

In [ ]:
MIN_TEZINA = 1.5

def deriviraj_ista_tema(min_tezina=MIN_TEZINA):
    neo4j_run("MATCH ()-[r:ISTA_TEMA {izvor:'auto'}]->() DELETE r")
    rezultat = neo4j_run('''
        // ukupan broj clanaka - za IDF
        MATCH (c:Clanak) WITH count(c) AS N

        // za svaki entitet: u koliko clanaka se pojavljuje
        MATCH (e:Entitet)<-[:SPOMINJE]-(c:Clanak)
        WITH N, e, count(DISTINCT c) AS df
        WHERE df >= 2
        WITH N, e, log(1.0 * N / df) AS idf

        // parovi clanaka iz razlicitih zakona, zbroj IDF-a dijeljenih entiteta
        MATCH (c1:Clanak)-[:SPOMINJE]->(e)<-[:SPOMINJE]-(c2:Clanak)
        WHERE c1.zakon_id < c2.zakon_id
        WITH c1, c2, sum(idf) AS tezina, count(DISTINCT e) AS zaj
        WHERE tezina >= $min_tezina AND zaj >= 2
        MERGE (c1)-[r:ISTA_TEMA {izvor:'auto'}]->(c2)
        SET r.tezina = tezina, r.zaj_entiteta = zaj
        RETURN count(r) AS n
    ''', {'min_tezina': min_tezina})
    n = rezultat[0]['n'] if rezultat else 0
    print(f'\u2713 Derivirano {n} ISTA_TEMA veza (IDF tezina >= {min_tezina}, medudokumentno)')
    return n

deriviraj_ista_tema()

_diag = neo4j_run('''
    MATCH (a:Clanak)-[:UPUCUJE_NA]->(b:Clanak)
    RETURN a.zakon_id = b.zakon_id AS unutar_istog, count(*) AS n
''')
print('\nUPUCUJE_NA raspodjela:')
for r in (_diag or []):
    oznaka = 'unutar istog zakona' if r['unutar_istog'] else 'MEDUDOKUMENTNO'
    print(f"  {oznaka:<22} {r['n']:>5}")

_most = neo4j_run('''
    MATCH (c1:Clanak)-[:SPOMINJE]->(e:Entitet)<-[:SPOMINJE]-(c2:Clanak)
    WHERE c1.zakon_id < c2.zakon_id
    RETURN count(DISTINCT e) AS mostovni_entiteti
''')
print(f"\nMostovnih entiteta (spajaju 2 zakona): "
      f"{_most[0]['mostovni_entiteti'] if _most else 0}")


✓ Derivirano 7 ISTA_TEMA veza (IDF tezina >= 1.5, medudokumentno)

UPUCUJE_NA raspodjela:
  unutar istog zakona      123

Mostovnih entiteta (spajaju 2 zakona): 28


## 8. Statistike grafa

Prikazujemo ukupan broj čvorova i relacija u Neo4j grafu.

In [ ]:
stats = neo4j_run('''
    MATCH (z:Zakon) WITH count(z) AS zakoni
    MATCH (c:Clanak) WITH zakoni, count(c) AS clanci
    OPTIONAL MATCH (s:Stavak)
    RETURN zakoni, clanci, count(s) AS stavci
''')
if stats:
    s = dict(stats[0])
    print('Graf statistike:')
    for k, v in s.items():
        print(f'  {k.capitalize()}: {v}')

per_doc = neo4j_run('''
    MATCH (z:Zakon)-[:SADRZI]->(c:Clanak)
    RETURN z.naziv AS zakon, count(c) AS clanaka,
           sum(CASE WHEN c.broj STARTS WITH 'pre' OR c.broj STARTS WITH 'chunk'
               THEN 1 ELSE 0 END) AS fallback
    ORDER BY zakon
''')
print('\nClanaka po dokumentu:')
for r in per_doc:
    print(f"  {r['zakon'][:55]:<57} {r['clanaka']:>4} (fallback: {r['fallback']})")

veze = neo4j_run('''
    MATCH ()-[r:UPUCUJE_NA]->()     RETURN 'UPUCUJE_NA'     AS tip, count(r) AS n
    UNION ALL
    MATCH ()-[r:RAZRADJUJE]->()     RETURN 'RAZRADJUJE'     AS tip, count(r) AS n
    UNION ALL
    MATCH ()-[r:ISTA_TEMA]->()      RETURN 'ISTA_TEMA'      AS tip, count(r) AS n
    UNION ALL
    MATCH ()-[r:SADRZI]->()         RETURN 'SADRZI'         AS tip, count(r) AS n
    UNION ALL
    MATCH ()-[r:IMA_STAVAK]->()     RETURN 'IMA_STAVAK'     AS tip, count(r) AS n
    UNION ALL
    MATCH ()-[r:SPOMINJE]->()       RETURN 'SPOMINJE'       AS tip, count(r) AS n
    UNION ALL
    MATCH ()-[r:SADRZI_OBVEZU]->()  RETURN 'SADRZI_OBVEZU'  AS tip, count(r) AS n
''')
print('\nRelacije u grafu:')
for r in veze:
    print(f"  {r['tip']:<25} {r['n']:>6} veza")

eksp = neo4j_run("MATCH ()-[r]->() WHERE r.izvor = 'ekspert' RETURN count(r) AS n")
print(f"\nEkspertnih veza (izvor='ekspert'): {eksp[0]['n'] if eksp else 0}")


Graf statistike:
  Zakoni: 4
  Clanci: 538
  Stavci: 1414

Clanaka po dokumentu:
  Pravilnik o temeljnom vojnom osposobljavanju                23 (fallback: 1)
  Ustav Republike Hrvatske 2018                              146 (fallback: 5)
  Zakon o obrani 2025                                        130 (fallback: 2)
  Zakon o sluzbi u Oruzanim snagama Republike Hrvatske 20    238 (fallback: 2)



Relacije u grafu:
  UPUCUJE_NA                   123 veza
  ISTA_TEMA                      7 veza
  SADRZI                       537 veza
  IMA_STAVAK                  1414 veza
  SPOMINJE                     421 veza
  SADRZI_OBVEZU                488 veza

Ekspertnih veza (izvor='ekspert'): 0


## 9. Učitavanje pitanja i odgovora

In [ ]:
try:
    with open('/content/drive/My Drive/diplomskiRad/pitanja_odgovori.json', 'r', encoding='utf-8') as f:
        qa_data = json.load(f)

    print(f"\u2713 Učitano {qa_data['metadata']['ukupno_pitanja']} pitanja i odgovora.")
    print(f"  Struktura:")
    for kategorija, info in qa_data['metadata']['struktura'].items():
        print(f"    - {kategorija}: {info['broj']} ({info['postotak']})")
except FileNotFoundError:
    print('Datoteka pitanja_odgovori.json nije pronađena na navedenoj putanji.')
    qa_data = None
except json.JSONDecodeError:
    print('Greška pri parsiranju JSON datoteke. Provjerite je li format ispravan.')
    qa_data = None

✓ Učitano 31 pitanja i odgovora.
  Struktura:
    - kompleksna_vise_dokumenata: 3 (9.67%)
    - kompleksna_jedan_dokument: 5 (16.13%)
    - jednostavna: 23 (74.2%)


## 10. GraphRAG retriever

Retriever pretražuje Neo4j vektorski indeks i dohvaća najsličnije članke za dani upit.

In [ ]:
def graph_retrieve_context(query: str, kategorija: str = None) -> str:
    """GraphRAG retriever — ispravke u odnosu na prvu verziju:
    1. Dohvat i po CLANCIMA i po STAVCIMA (uska pitanja pogadaju
       uski stavak, vraca se roditeljski clanak).
    2. Cap po dokumentu — garantirana pokrivenost svih akata kod
       medudokumentnih pitanja.
    3. Tipizirani traversal (UPUCUJE_NA | RAZRADJUJE | ISTA_TEMA +
       SPOMINJE preko zajednickog entiteta, samo medudokumentno)
       s tezinama po tipu veze umjesto fiksnog penaltija.
    4. Dedup po ID-u umjesto krhkog Jaccard preklapanja rijeci.
    5. top_k ovisan o kategoriji pitanja.
    """
    top_k = GRAPH_TOP_K.get(kategorija, GRAPH_TOP_K_DEFAULT)
    query_emb = embed_model.encode(['query: ' + query],
                                   normalize_embeddings=True)[0].tolist()

    res_cl = neo4j_run('''
        CALL db.index.vector.queryNodes('clanak_embedding', $k, $emb)
        YIELD node AS c, score
        WHERE c.tekst IS NOT NULL
        MATCH (z:Zakon)-[:SADRZI]->(c)
        RETURN c.id AS id, c.broj AS broj, c.tekst AS tekst,
               z.naziv AS zakon, score
    ''', {'k': GRAPH_VECTOR_POOL, 'emb': query_emb})

    res_st = neo4j_run('''
        CALL db.index.vector.queryNodes('stavak_embedding', $k, $emb)
        YIELD node AS s, score
        MATCH (c:Clanak)-[:IMA_STAVAK]->(s)
        WHERE c.tekst IS NOT NULL
        MATCH (z:Zakon)-[:SADRZI]->(c)
        WITH c, z, max(score) AS score
        RETURN c.id AS id, c.broj AS broj, c.tekst AS tekst,
               z.naziv AS zakon, score
    ''', {'k': GRAPH_VECTOR_POOL, 'emb': query_emb})

    seeds = {}
    for r in list(res_cl) + list(res_st):
        rec = {'id': r['id'], 'broj': r['broj'], 'tekst': r['tekst'] or '',
               'zakon': r['zakon'], 'score': float(r['score'])}
        if rec['id'] not in seeds or rec['score'] > seeds[rec['id']]['score']:
            seeds[rec['id']] = rec

    po_zakonu = {}
    for rec in sorted(seeds.values(), key=lambda x: x['score'], reverse=True):
        po_zakonu.setdefault(rec['zakon'], [])
        if len(po_zakonu[rec['zakon']]) < GRAPH_PER_DOC_CAP:
            po_zakonu[rec['zakon']].append(rec)
    seed_list = [rec for grupa in po_zakonu.values() for rec in grupa]
    seed_ids = [rec['id'] for rec in seed_list]
    seed_score = {rec['id']: rec['score'] for rec in seed_list}

    susjedi_res = neo4j_run('''
        MATCH (seed:Clanak) WHERE seed.id IN $ids

        OPTIONAL MATCH (seed)-[r:UPUCUJE_NA|RAZRADJUJE|ISTA_TEMA]-(n1:Clanak)
        WHERE n1.tekst IS NOT NULL
        OPTIONAL MATCH (zn1:Zakon)-[:SADRZI]->(n1)

        OPTIONAL MATCH (seed)-[:SPOMINJE]->(e:Entitet)<-[:SPOMINJE]-(n2:Clanak)
        WHERE n2.zakon_id <> seed.zakon_id AND n2.tekst IS NOT NULL
        OPTIONAL MATCH (zn2:Zakon)-[:SADRZI]->(n2)

        WITH seed,
             collect(DISTINCT {id: n1.id, broj: n1.broj, tekst: n1.tekst,
                               zakon: zn1.naziv, rel: type(r)})[0..4]  AS direktni,
             collect(DISTINCT {id: n2.id, broj: n2.broj, tekst: n2.tekst,
                               zakon: zn2.naziv, rel: 'SPOMINJE'})[0..3] AS entitetski
        RETURN seed.id AS sid, direktni + entitetski AS susjedi
    ''', {'ids': seed_ids})

    candidates = {rec['id']: dict(rec, rel='SEED') for rec in seed_list}
    for row in susjedi_res:
        base = seed_score.get(row['sid'], 0.0)
        for s in (row['susjedi'] or []):
            if not s or not s.get('id') or not s.get('tekst'):
                continue
            if s['id'] in candidates:
                continue
            w = GRAPH_REL_WEIGHTS.get(s.get('rel'), 0.7)
            candidates[s['id']] = {'id': s['id'], 'broj': s['broj'],
                                   'tekst': s['tekst'], 'zakon': s['zakon'],
                                   'score': base * w, 'rel': s.get('rel')}

    best = sorted(candidates.values(), key=lambda x: x['score'],
                  reverse=True)[:top_k]

    parts = []
    for c in best:
        tag = '' if c['rel'] == 'SEED' else f" | veza={c['rel']}"
        tekst = c['tekst'][:GRAPH_MAX_CHARS_PO_CLANKU]
        parts.append(f"[{c['zakon']}, Clanak {c['broj']} | "
                     f"score={c['score']:.2f}{tag}]\n{tekst}")
    return '\n\n---\n\n'.join(parts)

print('\u2713 GraphRAG retriever spreman (per-dokument + stavci + tipizirane veze)')


✓ GraphRAG retriever spreman (per-dokument + stavci + tipizirane veze)


## 11. Ekspertne veze

Rucno kurirane medudokumentne veze — sloj "ekspertnog truda" (GraphRAG-ekspert).
Tipovi definirani unaprijed: `RAZRADJUJE` (nizi akt razraduje visi) i
`ISTA_TEMA` (ista materija u razlicitim aktima). Svaka veza nosi
`izvor: 'ekspert'` i `pitanje_id` radi sljedivosti i lakog uklanjanja
(usporedba GraphRAG-auto vs GraphRAG-ekspert).

**Napomena:** popuniti listu nakon pregleda pravnog eksperta (pitanja 32-43).
ID format mora odgovarati parseru: `{zakon_id}_cl_{broj}` — provjeriti kako
su zapisani brojevi tipa "21.m" (npr. `21m`).

In [ ]:
EKSPERTNE_VEZE = [
]

def unesi_ekspertne_veze(veze):
    ok, fail = 0, []
    for src, dst, tip, pid in veze:
        if tip not in ('RAZRADJUJE', 'ISTA_TEMA'):
            fail.append((src, dst, f'nepoznat tip {tip}')); continue
        res = neo4j_run(f'''
            MATCH (a:Clanak {{id: $src}})
            MATCH (b:Clanak {{id: $dst}})
            MERGE (a)-[r:{tip} {{izvor: 'ekspert', pitanje_id: $pid}}]->(b)
            RETURN a.id AS a, b.id AS b
        ''', {'src': src, 'dst': dst, 'pid': pid})
        if res: ok += 1
        else:   fail.append((src, dst, 'cvor nije pronaden'))
    print(f'\u2713 Uneseno {ok}/{len(veze)} ekspertnih veza')
    for f in fail:
        print(f'  \u2717 {f}')

if EKSPERTNE_VEZE:
    unesi_ekspertne_veze(EKSPERTNE_VEZE)
else:
    print('(lista EKSPERTNE_VEZE je prazna — GraphRAG-auto rezim)')


(lista EKSPERTNE_VEZE je prazna — GraphRAG-auto rezim)


## 12. Dijagnostika konteksta

Provjera doprinosi li graf stvarno kontekstu: koliko clanaka dolazi iz
grafa (veza=...) naspram vektorski (SEED) i koliko razlicitih zakona
kontekst pokriva. Pokrenuti na 2-3 kompleksna pitanja **prije i poslije**
izmjena — razlika je izravan dokaz za rad.

In [ ]:
def dijagnoza_konteksta(query, kategorija='kompleksno_vise_dokumenata'):
    ctx = graph_retrieve_context(query, kategorija=kategorija)
    blokovi = ctx.split('\n\n---\n\n') if ctx else []
    iz_grafa = sum(1 for b in blokovi if '| veza=' in b)
    zakoni = set()
    for b in blokovi:
        header = b.split(']')[0]
        zakoni.add(header.split(',')[0].strip('['))
    print(f'Pitanje: {query[:70]}...')
    print(f'  Clanaka u kontekstu: {len(blokovi)}')
    print(f'  Iz grafa (veze):     {iz_grafa}')
    print(f'  Pokriveno zakona:    {len(zakoni)} -> {sorted(zakoni)}')
    return ctx

_ = dijagnoza_konteksta('Kako se povezuju prigovor savjesti iz Ustava, '
                        'civilna sluzba i sustav vojne obveze?')
print()
_ = dijagnoza_konteksta('Povezite ustavnu ulogu Predsjednika kao vrhovnog '
                        'zapovjednika s lancem zapovijedanja nacelnika Glavnog stozera.')


Pitanje: Kako se povezuju prigovor savjesti iz Ustava, civilna sluzba i sustav ...
  Clanaka u kontekstu: 12
  Iz grafa (veze):     2
  Pokriveno zakona:    4 -> ['Pravilnik o temeljnom vojnom osposobljavanju', 'Ustav Republike Hrvatske 2018', 'Zakon o obrani 2025', 'Zakon o sluzbi u Oruzanim snagama Republike Hrvatske 2025']



Pitanje: Povezite ustavnu ulogu Predsjednika kao vrhovnog zapovjednika s lancem...
  Clanaka u kontekstu: 12
  Iz grafa (veze):     0
  Pokriveno zakona:    4 -> ['Pravilnik o temeljnom vojnom osposobljavanju', 'Ustav Republike Hrvatske 2018', 'Zakon o obrani 2025', 'Zakon o sluzbi u Oruzanim snagama Republike Hrvatske 2025']


## 13. Testiranje — GraphRAG pristup

GraphRAG pristup koristi Neo4j vektorski indeks za retrieval relevantnih članaka zakona, a zatim Qwen2.5-14B-Instruct generira odgovor na temelju pronađenog konteksta.

### GraphRAG chat funkcija
Funkcija `graph_rag_chat` dohvaća kontekst preko `graph_retrieve_context` (Neo4j vektorski indeks + graf veze, s budžetom po kategoriji pitanja definiranim u `GRAPH_TOP_K`) i umeće ga u `SYSTEM_PROMPT` prije slanja modelu. Koristi
  iste generacijske parametre kao Vanilla i RAG (512 tokena, temperatura 0.3, deterministicko dekodiranje) radi izravne usporedivosti rezultata.

In [ ]:
def graph_rag_chat(user_question, kategorija=None):
    context = graph_retrieve_context(user_question, kategorija=kategorija)
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT + '\n\nKontekst za odgovor:\n' + context},
        {'role': 'user', 'content': f'{user_question}\n\nOdgovori NA HRVATSKOM JEZIKU koristeci LATINICU.'}
    ]
    return generate_response(messages, max_new_tokens=512, temperature=0.3, do_sample=False)


### Testiranje GraphRAG pristupa — sva pitanja

Prolazimo kroz sva pitanja iz JSON datoteke, generiramo odgovore GraphRAG pristupom i rezultate nadopunjujemo u postojeće redove CSV datoteke kao nove stupce.

In [ ]:
if qa_data is None:
    print('QA datoteka nije ucitana.')
else:
    df_rezultati = pd.read_csv(csv_path, encoding='utf-8-sig')

    KATEGORIJA_MAP = {
        'jednostavna_pitanja':                'jednostavno',
        'kompleksna_pitanja_jedan_dokument':  'kompleksno_jedan_dokument',
        'kompleksna_pitanja_vise_dokumenata': 'kompleksno_vise_dokumenata',
    }

    all_questions = []
    all_categories = []
    for key, value in qa_data.items():
        if key != 'metadata' and isinstance(value, list):
            for item in value:
                all_questions.append(item)
                all_categories.append(KATEGORIJA_MAP.get(key, key))

    graph_rag_odgovori = []
    graph_rag_vremena = []

    for i, (qa_item, kategorija) in enumerate(zip(all_questions, all_categories)):
        user_question = qa_item.get('pitanje', qa_item.get('question', ''))
        ocekivani = qa_item.get('odgovor', qa_item.get('answer', ''))

        print(f"\n{'='*80}")
        print(f'Pitanje {i + 1}/{len(all_questions)}: {user_question}')
        print(f'Kategorija: {kategorija} (top_k={GRAPH_TOP_K.get(kategorija, GRAPH_TOP_K_DEFAULT)})')
        print(f'Ocekivani odgovor: {ocekivani}')
        print(f"{'='*80}")

        answer, vrijeme = graph_rag_chat(user_question, kategorija=kategorija)

        graph_rag_odgovori.append(answer)
        graph_rag_vremena.append(round(vrijeme, 2))

    df_rezultati['graph_rag_odgovor'] = graph_rag_odgovori
    df_rezultati['graph_rag_vrijeme_sekunde'] = graph_rag_vremena
    df_rezultati.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f"\n{'='*80}")
    print(f'GraphRAG rezultati nadopunjeni u: {csv_path}')
    print(f'Ukupno obradeno pitanja: {len(graph_rag_odgovori)}')
    print(f"Prosjecno vrijeme po pitanju: {sum(graph_rag_vremena)/len(graph_rag_vremena):.2f} sekundi")



Pitanje 1/31: Opišite cjelokupnu proceduru transformacije novaka u razvrstanog pričuvnika kroz sve pravne akte koji reguliraju taj proces.
Kategorija: kompleksno_vise_dokumenata (top_k=12)
Ocekivani odgovor: Procedura je regulirana kroz 4 pravna akta. 1. USTAVNI TEMELJ (Ustav RH, čl. 47). Vojna obveza je dužnost svih za to sposobnih državljana. Dopušten prigovor savjesti iz vjerskih i/ili moralnih razloga uz obvezu ispunjavanja drugih dužnosti određenih zakonom. 2. NOVAČKA OBVEZA (Zakon o obrani, čl. 19-21). Nastaje u kalendarskoj godini kad državljanin RH navrši 18 godina. Državljanin RH uvodi se u vojnu evidenciju u kalendarskoj godini u kojoj navrši 18 godina života. Prolazi zdravstveni pregled i psihologijsko ispitivanje. 3. TEMELJNO VOJNO OSPOSOBLJAVANJE (Zakon o obrani čl. 21.m, Pravilnik o temeljnom vojnom osposobljavanju). Traje 2 mjeseca. Ministar obrane donosi Plan upućivanja u tekućoj godini do 15.12. za iduću godinu. Odluku o upućivanju donosi 40 dana prije upućivanja kand

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenizati

Generirani odgovor:
Cjelokupna procedura transformacije novaka u razvrstanog pričuvnika regulirana je slijedećim pravnim aktima:  

1. **Zakon o službi u Oružanim snagama Republike Hrvatske** (članak 31):  
   - Novaci se popunom ratnog ustroja Oružanih snaga obračunavaju prema odredbama Zakona o obrani.  
   - Ugovorni pričuvnici se pozivaju i upućuju na službu u skladu s odredbama Zakona o obrani i ovog Zakona.  

2. **Zakon o obrani** (članak 21.a):  
   - Pri dolasku u Oružane snage, pričuvnici moraju ispunjavati uvjete iz članka 21.a, uključujući putovanje od prebivališta do mjesta službe i natrag.  

3. **Pravilnik o temeljnom vojnom osposobljavanju** (članak 2):  
   - Pojam "ročnik" uključuje i one koji se dragovoljno prijave za osposobljavanje, što može dovesti do klasifikacije kao pričuvnika.  

4. **Postupak prijave i evidencije kandidata** (Pravilnik o temeljnom vojnom osposobljavanju, članak 1):  
   - Kandidati se prijavljuju i evidentiraju prema propisanom postupku, što 

[transformers] Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generirani odgovor:
Hierarhija odlučivanja o uporabi Oružanih snaga u različitim situacijama regulirana je kroz **Ustav Republike Hrvatske** i **Zakon o obrani** kako slijedi:  

### **Prema Ustavu (Članak 7):**  
- **Ratno stanje / Teritorijalna cjelovitost:**  
  - Odluku o upotrebi Oružanih snaga donose **Hrvatski sabor**, **Predsjednik Republike**, **predsjednik Vlade** i **ministar obrane** zajedno, u skladu s Ustavom i Zakonom o obrani.  
  - Ako Predsjednik odbije suglasnost, Hrvatski sabor donosi odluku **dvotrećinskom većinom**.  

- **Međunarodni ugovori / Saveznička pomoć:**  
  - Ulazak stranih Oružanih snaga u RH moguć je samo na temelju **međunarodnih ugovora** i odluke Hrvatskog sabora (uz suglasnost Predsjednika).  

- **Protupožarna zaštita / Spašavanje:**  
  - Oružane snage mogu pomagati vatrogastvu i civilnoj zaštiti po odluci Vlade.  

### **Prema Zakonu o obrani (Članak 46 & 57):**  
- **Borbeno djelovanje protiv civilnih zrakoplova:**  
  - Regulirano posebnim za

...

Generirani odgovor:
Djelatnoj vojnoj osobi u slučaju rođenja djeteta pripada **sedam radnih dana** plaćenog dopusta, kako je definirano u [Zakon o službi u Oružanim snagama Republike Hrvatske 2025, Članak 159](https://www.narodne-novine.hr/#_133086). Ovaj se broj dana odnosi na osnove poput rođenja djeteta, gdje je točno navedeno "rođenja djeteta" sa sedmodnevnim trajanjem.
Ukupno vrijeme za odgovor: 14.96 sekundi

Pitanje 30/31: Koliko dana plaćenog dopusta pripada djelatnoj vojnoj osobi u slučaju zaključenja braka?
Kategorija: jednostavno (top_k=5)
Ocekivani odgovor: Djelatna vojna osoba ima pravo na plaćeni dopust od sedam radnih dana u slučaju zaključenja braka.


[transformers] Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generirani odgovor:
Djelatnoj vojnoj osobi u slučaju zaključenja braka pripada **sedam radnih dana** plaćenog dopusta, kako je definirano u **Članku 159.** Zakona o službi u Oružanim snagama Republike Hrvatske.  

Ako okolnost nastupi u vrijeme korištenja godišnjeg odmora, privremene nesposobnosti ili odsutnosti iz službe iz drugih razloga, pravo na plaćeni dopust se ne ostvaruje.
Ukupno vrijeme za odgovor: 14.10 sekundi

Pitanje 31/31: Što je vojna iskaznica i tko ju izdaje?
Kategorija: jednostavno (top_k=5)
Ocekivani odgovor: Vojna iskaznica je javna isprava kojom vojni obveznik dokazuje identitet dok izvršava vojnu obvezu. Izdaje ju nadležni područni odjel za poslove obrane koji vojnog obveznika vodi u evidenciji.


[transformers] Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generirani odgovor:
Vojna iskaznica je identifikacijski dokument koji dokazuje pripadnost Oružanim snagama Republike Hrvatske. Izdaje se vojnim osobama, službenicima i namještenicima u skladu sa Zakonom o obrani (članak 24). Ministar obrane propisuje oblik i sadržaj ove iskaznice pravilnikom. Ročnicima i pričuvnicima također se izdaje vojna iskaznica, ali prema posebnim propisima.
Ukupno vrijeme za odgovor: 12.66 sekundi

GraphRAG rezultati nadopunjeni u: /content/drive/My Drive/diplomskiRad/rezultati.csv
Ukupno obradeno pitanja: 31
Prosjecno vrijeme po pitanju: 24.63 sekundi
